<a href="https://colab.research.google.com/github/alfianfawwaz00-tech/proyek-analisis-krisis-2/blob/main/P_ADRO.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers pandas-ta scikit-learn requests yfinance openpyxl

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.3/240.3 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 96.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.5/16.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 MB 16.6 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: llvmlite
    Found existing installation: llvmlite 0.43.0
    Uninstalling llvmlite-0.43.0:
      Successfully uninstalled llvmlite-0.43.0
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninst

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import requests
from transformers import pipeline
import pandas_ta as ta
from sklearn.linear_model import LinearRegression

api_key = "32277d79e9ac4b689e2cce4041252318"
url = "https://newsapi.org/v2/everything?q=saham&language=id&apiKey=" + api_key

response = requests.get(url)
data = response.json()
articles = data.get('articles', [])
berita_list = []

for article in articles:
  judul = article.get('title')
  tanggal = article.get('publishedAt')[:10]
berita_list.append({'tanggal': tanggal, 'judul': judul})

df_berita = pd.DataFrame(berita_list)

model_ai = pipeline("sentiment-analysis", model="w11wo/indonesian-roberta-base-sentiment-classifier")
daftar_sentimen = []

for teks in df_berita['judul']:
  hasil = model_ai(teks)
  label = hasil[0]['label']
if label == 'positive':
  skor = 1
elif label == 'negative':
  skor = -1
else:
  skor = 0
daftar_sentimen.append(skor)

df_berita['skor_sentimen'] = daftar_sentimen
df_sentimen_harian = df_berita.groupby('tanggal')['skor_sentimen'].mean().reset_index()
df_sentimen_harian['tanggal'] = pd.to_datetime(df_sentimen_harian['tanggal'])

saham = yf.Ticker("ADRO.JK")
df_saham = saham.history(start="2026-01-01", end="2026-07-19")
df_saham.reset_index(inplace=True)
df_saham['Date'] = pd.to_datetime(df_saham['Date']).dt.tz_localize(None)

df_saham['SMA_10'] = ta.sma(df_saham['Close'], length=10)
df_saham['RSI_14'] = ta.rsi(df_saham['Close'], length=14)
df_saham.dropna(inplace=True)

df_gabungan = pd.merge(df_saham, df_sentimen_harian, left_on='Date', right_on='tanggal', how='left')
df_gabungan['skor_sentimen'] = df_gabungan['skor_sentimen'].fillna(0)

X = df_gabungan[['SMA_10', 'RSI_14', 'skor_sentimen', 'Volume']]
y = df_gabungan['Close']

model_regresi = LinearRegression()
model_regresi.fit(X, y)
df_gabungan['Prediksi_Harga'] = model_regresi.predict(X)

df_gabungan.to_excel('Hasil_Analisis_KTI.xlsx', index=False)
print("Program selesai")

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  499MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/328 [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/467k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.38M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/yfinance/scrapers/history.py:204: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()


Program selesai
